# 03 — Income finale: combining volume and price

The two outcomes are not independent stories — they multiply into the quantity that actually matters to a farmer, **revenue**. They combine on an exact identity:

$$\log(\text{income}) = \log(\text{quantity}) + \log(\text{price})$$

This notebook estimates **nothing new**. The causal work is already done in `01_data` (volume) and `02_price` (price); here we simply **recombine** their per-country synthetic-control effects on that identity. The key property is that the identity holds **country by country**: each effect is a country's own deviation from its own counterfactual, so the fact that the two analyses used *different control sets* (the UK and others absent from price; Ireland dropped from quantity) does not matter — only the treated countries present in **both** valid sets enter.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROC = Path("data/processed")
CUTOFF = pd.Timestamp("2015-04-01")

## 1. Load the two effect sets

Each file is the per-country, per-month synthetic-control gap (treated − synthetic, in logs) from its analysis, restricted to the valid set that passed the pre-RMSPE cut. Income is defined only on the **intersection** of treated countries present in both.

In [ ]:
gaps_quantity = pd.read_parquet(PROC / "gaps_quantity.parquet")
gaps_price    = pd.read_parquet(PROC / "gaps_price.parquet")

countries = sorted(set(gaps_quantity.columns) & set(gaps_price.columns))
dropped = sorted((set(gaps_quantity.columns) | set(gaps_price.columns)) - set(countries))
print("income countries:", countries, f"({len(countries)})")
print("dropped (not in both valid sets):", dropped or "none")

## 2. The decomposition

Because income is the *product* of quantity and price, its synthetic-control counterfactual is the product of the two outcome counterfactuals, and the gaps **add exactly in logs**:

$$\text{gap}_{\text{income}} = \text{gap}_{\text{quantity}} + \text{gap}_{\text{price}}$$

We average each over the post-period, restricted to the months where **both** outcomes are observed, so the identity holds exactly in the table (Luxembourg, for instance, has no 2019 price, so those months drop from all three columns together).

**One interpretive guard, stated up front.** The price column is a *statistical null* — four methods agreed there is no robust treated-vs-control price differential, and the small per-country price entries (especially Cyprus's) are fit noise, not a genuine relative price gain. So the income effect should be read as **driven by volume**, with the price contribution ≈ 0 within noise. We show the decomposition because the arithmetic is exact and honest; we do not oversell the price column as a real bonus.

In [ ]:
# Restrict to months where both outcomes are observed, so income = quantity + price exactly.
gq, gp = gaps_quantity[countries], gaps_price[countries]
both = gq.notna() & gp.notna()
gq, gp = gq.where(both), gp.where(both)
gi = gq + gp

post = lambda g: g[g.index >= CUTOFF].mean()      # per-country post-period mean
table = pd.DataFrame({"quantity": post(gq), "price": post(gp), "income": post(gi)})
table = table.sort_values("income", ascending=False)

table_pct = (np.exp(table) - 1) * 100            # log gaps -> % for readability

print("Per-country effect (log points):")
print(table.round(3))
print("\nPer-country effect (%):")
print(table_pct.round(1))
print("\nAverage across countries (%):")
print(table_pct.mean().round(1))

In [ ]:
# Stacked decomposition: volume + price = income, per country (log points add exactly).
order = table.index
q = table.loc[order, "quantity"] * 100
p = table.loc[order, "price"] * 100

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(order, q, color="#378ADD", label="volume")
ax.barh(order, p, left=q, color="#D85A30", label="price")
ax.plot(table.loc[order, "income"] * 100, order, "D", color="black", ms=5,
        label="income (total)")
ax.axvline(0, color="grey", lw=1)
ax.set(title="Farm income decomposition: volume vs price (post-2015, vs synthetic)",
       xlabel="effect (log points x100 ~ %)")
ax.legend(); plt.tight_layout(); plt.show()

## Conclusions — what the abolition did to farm income

_Fill the bracketed figures from the table above once run._

Relative to the control group, farm-gate **income moved essentially with volume**: the constrained countries expanded output by **[~+X%]**, and the integrated EU price market spared them any *relative* price penalty, so the volume gain passed through to income largely intact — an average income effect of **[~+Y%]** across the [N] countries in both valid sets.

| Channel | Effect (vs synthetic) | Reading |
|---|---|---|
| **Volume** | **[+X%]** | The real, robust effect — quota lifted, constrained countries grew |
| **Price** | **[~0%]** | Null differential (four methods); fit noise, not a relative gain |
| **Income** | **[+Y%]** ≈ volume | Pass-through of the volume effect, price-neutral vs control |

**The economic story, in one line:** the abolition let the bound countries produce more, and because milk prices form at the EU level, doing so did **not** cost them a relative price cut — so more milk meant more income, roughly one-for-one against the control group.

**Honest scope, restated.** This is an effect *relative to the control group*, not an absolute change: the EU-wide price may have fallen for everyone (we cannot see that — no counterfactual for the whole EU), so the *absolute* income gain could be smaller than the relative one. Ireland is absent (it did not pass the quantity pre-RMSPE cut), and Luxembourg's post-period ends in 2018. Within those limits, the decomposition is exact and method-consistent: both channels come from the same augmented synthetic control, combined on an identity that holds country by country.